# The rule and the forest

I compare a rule I wrote by hand with a random forest. Both see the same five features and score the same pages.

Every number here comes from `reports/results.json`. `python -m bleedtracker reproduce` builds that file from the warehouse and checks it against my original results.

In [1]:
import pandas as pd

from bleedtracker.results import load_results

results = load_results()

## The label

A page is declining when its impressions next month are more than 20% below this month's. It's a proxy. A 21% drop and a 19% drop aren't different events. A page that vanishes from Search Console counts as a total loss. The data can't tell a deleted page from one that lost all its traffic.

## The five features

Every feature is measured inside the feature month. All five are known on its last day.

| Feature | Built from |
|---|---|
| `log_impressions` | the log of 1 plus the month's impressions |
| `active_days` | days with at least one impression |
| `position` | the average position, weighted by impressions |
| `ctr` | clicks divided by impressions |
| `momentum` | second-half impressions divided by first-half impressions |

The first half gets the extra day in a month with an odd number of days.

## The rule

Before any model existed, I wrote a score by hand and froze it.

```
severity   = clip(1 - momentum / 0.8, 0, 1)
underclick = 1 when ctr is below the median CTR of the page's position band
reachable  = 1 when position is between 1 and 20
score      = severity × log1p(impressions) × (1 + underclick) × (1 + 0.5 × reachable)
```

The weights are round numbers a person can argue with. The only fitted part is the median CTR of each position band. It's learned on training clients only. These are the medians from the first deployment, in percent.

In [2]:
anchor = results["anchor"]
pd.Series(anchor["band_median_ctr_percent"], name="median CTR (%)").to_frame()

,median CTR (%)
up to 3,0.1775
3 to 10,0.1889
10 to 20,0.1165
over 20,0.0000


Each page also gets a reason code from the same three signals. A page that isn't sliding is holding steady. A sliding page is underclicking, reachable or deep.

In [3]:
pd.Series(anchor["reason_codes_in_slice"], name="pages in March 2026").map("{:,}".format).to_frame()

,pages in March 2026
holding_steady,"57,668"
slipping_and_underclicking,"12,292"
slipping_reachable_page,"8,357"
slipping_deep_page,"7,136"


## The forest

The forest has 300 trees, at least 400 pages per leaf and seed 42. I picked it with cross-validation inside the training clients, before I looked at the held-out clients. It sees the same five features as the rule. Nothing about it is tuned again later.

## The first deployment

The split is by client. 18 clients train and 9 are held out. All of a client's pages land on one side. Pages of one client share a template, a CMS and an editorial calendar. A split by page would let the forest learn a client and call it skill.

In [4]:
pd.Series(
    {
        "pages in March 2026": f"{anchor['slice_pages']:,}",
        "clients": anchor["slice_clients"],
        "training pages": f"{anchor['train_pages']:,}",
        "held-out pages": f"{anchor['test_pages']:,}",
        "held-out base rate": f"{anchor['test_base_rate']:.1%}",
        "rule precision@50": anchor["rule_precision_at_50"],
        "forest precision@50": anchor["forest_precision_at_50"],
        "forest AUC": anchor["forest_auc"],
        "pages the rule scores zero": f"{anchor['rule_zero_pages']:,}",
        "declines among them": f"{anchor['rule_zero_declines']:,}",
    },
    name="March 2026, held-out clients",
).to_frame()

,"March 2026, held-out clients"
pages in March 2026,"85,453"
clients,27
training pages,"59,682"
held-out pages,"25,771"
held-out base rate,49.4%
rule precision@50,0.9
forest precision@50,0.9
forest AUC,0.6532
pages the rule scores zero,"16,556"
declines among them,"6,746"


In March 2026 the rule and the forest each put 45 declining pages in the first 50. The held-out base rate was 49.4%. The rule scores 16,556 pages exactly zero. Those pages hold 6,746 declines the rule can't rank at all. The shadow queue in notebook 04 is built for them.

In [5]:
pd.Series(anchor["rule_precision_at_k"], name="rule precision").rename_axis("K").to_frame()

,rule precision
K,
10,0.800
25,0.880
50,0.900
100,0.910
250,0.912
